# Load the data

The data consists of:

- `sensors` - densely placed low cost sensor data
- `official` - sparse refeference measurements
- `mesh` - a mesh on which sensor data will be placed
- `predict` - a mesh on which final predictions will be made

In [ ]:
import xarray as xr

# load
official = xr.open_zarr('official.zarr/')
sensors = xr.open_zarr('sensors.zarr/')
mesh = xr.open_zarr('mesh.zarr/')
predict = xr.open_zarr('predict.zarr/')

# print out
for ds in official, sensors, mesh, predict:
    print(50*'-'+f'\n\n{ds}\n')

# Plotting the data

In [ ]:
# add more interactivity
%matplotlib inline

## Spatially

In [ ]:
# select a random time
import numpy as np
import matplotlib.pyplot as plt
time = np.random.choice(predict.time, 1)

# plot the whole domain
plt.close()
predict.sel(time=time).plot.scatter(x='x', y='y', s=10, c='k', label='mesh')
sensors.sel(time=time).plot.scatter(x='x', y='y', s=15, c='C1', label='senors')
official.sel(time=time).plot.scatter(x='x', y='y', s=25, c='C0', label='official')
plt.gca().set_aspect('equal', 'box')
plt.gcf().set_size_inches(8, 8)
plt.legend()
plt.show()

## Time series

In [ ]:
# select a random reference station by id
sample_id = np.random.choice(official.id)

# subsets according to the selected station neighbourhood
sample_official = official.sel(id=sample_id).compute()
sample_sensors = sensors.where(lambda ds: (ds.x - sample_official.x) < 5000)\
                        .where(lambda ds: (ds.y - sample_official.y) < 5000)\
                        .dropna(dim='id')\
                        .compute()

# plot
plt.close()
sample_sensors.calib.sel(id=np.random.choice(sample_sensors.id, 10)).plot(hue='id', alpha=0.1, c='C1', add_legend=False)
sample_sensors.calib.median(dim='id').plot(c='C1')
sample_official.pm25.plot(c='C0')
plt.show()

## As an image

In [ ]:
# select a sub-domain @ (official id, time)
sample_id = np.random.choice(official.id)
sample_time = np.random.choice(official.time)

# define image properties
image_size = 16
patch_size = 8

# select data within domain
sample_official = official.sel(time=sample_time, id=sample_id, drop=True)

sample_mesh = mesh.where(np.abs(mesh.x - sample_official.x) < image_size*1000/2) \
                  .where(np.abs(mesh.y - sample_official.y) < image_size*1000/2) \
                  .dropna(dim='id')

sample_sensors = sensors.sel(time=sample_time, drop=True)\
                        .where(lambda ds: ds.id_mesh.isin(sample_mesh.id))\
                        .dropna(dim='id')

sample_image = sample_sensors.compute()\
                             .set_coords('id_mesh')\
                             .groupby('id_mesh').mean()\
                             .rename({'id_mesh': 'id'})\
                             .drop_vars(['x','y'])\
                             .merge(sample_mesh)\
                             .set_coords(['y','x'])\
                             .set_index(id=['y','x'])\
                             .unstack('id')

# print
print(50*'-'+f'\n\n{sample_image}\n')

# plot
plt.close()

sample_mesh.plot.scatter(x='x', y='y', s=5, c='k')
sample_image.calib.plot()
sample_official.expand_dims('point').plot.scatter(x='x', y='y', s=40)
sample_sensors.plot.scatter(x='x', y='y', s=40, c='C1')

plt.gca().set_aspect('equal','box')
plt.gca().set_xticks(sample_image.x[::patch_size]-500)
plt.gca().set_yticks(sample_image.y[::patch_size]-500)
plt.gca().grid(True)
plt.show()

# Prepairing the data & model

## `Dataset`

In [ ]:
import dask
dask.config.set(scheduler='sync')

from torch import FloatTensor
from torch.utils.data import Dataset

class MyDataset(Dataset):
    
    def __init__(self):

        # input data
        self.official = official
        self.sensors = sensors
        self.mesh = mesh

        # parameters
        self.image_size = 16

        # stack official's coordinates to get individual samples
        self.official = self.official.stack(sample=['id','time'])

    def __len__(self):
        return self.official.sample.size

    def __getitem__(self, sample):

        sample_official = self.official.isel(sample=sample)

        sample_mesh = self.mesh.where(np.abs(self.mesh.x - sample_official.x) < self.image_size*1000/2)\
                               .where(np.abs(self.mesh.y - sample_official.y) < self.image_size*1000/2)\
                               .dropna(dim='id')
        
        sample_sensors = self.sensors.sel(time=sample_official.time.values)\
                                     .where(lambda ds: ds.id_mesh.isin(sample_mesh.id))\
                                     .dropna(dim='id')

        sample_image = sample_sensors.compute()\
                                     .set_coords('id_mesh')\
                                     .groupby('id_mesh').mean()\
                                     .rename({'id_mesh': 'id'})\
                                     .drop_vars(['x','y'])\
                                     .merge(sample_mesh)\
                                     .set_coords(['y','x'])\
                                     .set_index(id=['y','x'])\
                                     .unstack('id')

        # assign a mask where values exist
        sample_image['mask'] = sample_image.calib.notnull()

        # fill nan's with zero
        sample_image = sample_image.fillna(0)

        # convert to PyTorch tensors
        tensor_image = FloatTensor(sample_image.to_array().to_numpy())
        tensor_official = FloatTensor(sample_official.pm25.to_numpy()).reshape(-1)

        # return
        return tensor_image, tensor_official

# define
dataset = MyDataset()

# sample & print shapes
idx = np.random.choice(len(dataset))
tensor_image, tensor_official = dataset[idx]
print(50*'-'+f'\n\n{tensor_image.shape}\n')
print(50*'-'+f'\n\n{tensor_official.shape}\n')

# plot
plt.close()
_, axs = plt.subplots(1, 4, figsize=(13, 3), sharex=True, sharey=True)
for iax, ax in enumerate(axs.flatten()):
    ax.pcolor(tensor_image[iax,...])
plt.show()

## `DataLoader`

In [ ]:
from torch.utils.data import random_split, DataLoader

# split the dataset
my_dataset = MyDataset()
train_dataset, val_dataset, test_dataset = random_split(my_dataset, [0.6, 0.2, 0.2])

# create dataloaders
batch_size = 8
train_dataloader = DataLoader(train_dataset, batch_size, num_workers=2, shuffle=True)
val_dataloader = DataLoader(train_dataset, batch_size, num_workers=2)
test_dataloader = DataLoader(test_dataset, batch_size, num_workers=2)

# print a train batch
batch_image, batch_official = next(iter(train_dataloader))
print(50*'-'+f'\n\n{batch_image.shape}\n')
print(50*'-'+f'\n\n{batch_official.shape}\n')

# Training the model

## `LightningModule`

In [ ]:
from torch.nn import MSELoss
from torch.optim import Adam

from mfai.pytorch.models.vit import ViTClassifier, ViTClassifierSettings
from mfai.pytorch.lightning_modules import SegmentationLightningModule


class MyLightningModule(SegmentationLightningModule):

    def __init__(self):

        # get a batch & define image properties
        image_batch, official_batch = next(iter(train_dataloader))
        _, in_channels, *input_shape = image_batch.shape
        _, out_channels = official_batch.shape

        # define model
        settings = ViTClassifierSettings(patch_size=8)
        arch = ViTClassifier(in_channels, out_channels, input_shape, settings)
        type_segmentation = 'regression'
        loss = MSELoss()
        super().__init__(arch, 'regression', loss)

    # redefine optimizer
    def configure_optimizer(self):
        return Adam(self.parameters(), lr=1e-3)

    # remove image plotting at validation step
    def val_plot_step(*args):
        return

# make some test predictions & print results
model = MyLightningModule()
prediction_batch = model(batch_image)

print(50*'-'+f'\n\n{prediction_batch}\n')
print(50*'-'+f'\n\n{prediction_batch.shape}\n')

## Logging

In [ ]:
from lightning.pytorch.loggers import TensorBoardLogger

# logger
logger = TensorBoardLogger(save_dir='lightning_logs',
                           name='train',
                           version='')

# remove previous log files
!rm -rf lightning_logs/train

# enable the logger UI
%load_ext tensorboard
%tensorboard --logdir lightning_logs/train

## `Trainer.fit()`

In [ ]:
from lightning import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint

# reload the tensorboard
%reload_ext tensorboard

# callbacks
callbacks = [
    ModelCheckpoint(monitor='val_loss',
                    mode='min',
                    save_top_k=1),
]

# define trainer & fit
trainer = Trainer(logger=logger,
                  callbacks=callbacks,
                  max_epochs=10,
                  limit_train_batches=24,
                  limit_val_batches=24,
                  log_every_n_steps=1)

trainer.fit(model, train_dataloader, val_dataloader)

## `Trainer.test()`

In [ ]:
# subset datasets & concat them together
from torch.utils.data import ConcatDataset, Subset

# dataset & dataloader
n_samples = 24
concat_dataset = ConcatDataset([
    Subset(train_dataset, np.arange(n_samples)),
    Subset(val_dataset, np.arange(n_samples)),
    Subset(test_dataset, np.arange(n_samples)),
])
concat_dataloader = DataLoader(concat_dataset, batch_size=1, num_workers=2)

# test the model performance on the samples
_ = trainer.test(model, concat_dataloader)

In [ ]:
# load the trainer.test()
import pandas as pd
import seaborn as sns
loss = pd.read_csv('lightning_logs/train/metrics_test_set.csv', usecols=['loss'])

# assign 'split' column
loss.loc[:n_samples,'split'] = 'train'
loss.loc[n_samples:2*n_samples,'split'] = 'val'
loss.loc[2*n_samples:,'split'] = 'test'

# plot average errors across samples
plt.close()
sns.barplot(loss, y='loss', hue='split')
plt.show()

# Evaluating the model

## Logging & hyper-parameters

In [ ]:
# show the hparams.yaml file
!echo '---------------'
!cat lightning_logs/train-full/hparams.yaml
!echo '---------------'

# load the full training tensorboard log
%load_ext tensorboard
%tensorboard --logdir lightning_logs/train-full

## Time series

In [ ]:
# load the inferences from the fully trained model on the reference data
official_inferences = xr.open_zarr('official-inferences.zarr/')

# merge the inferred ones with the original observations
obs = official.pm25.rename('obs')
pred = official_inferences.pm25.rename('pred')
merged = xr.merge([obs, pred]).to_array()

# print
print(50*'-'+f'\n\n{merged}\n')

# plot
%matplotlib inline
plt.close()
merged.plot(col='id', col_wrap=5, hue='variable')
plt.show()

## Spatially

In [ ]:
# load the inferences from the fully trained model on the prediction mesh
predict_inferences = xr.open_zarr('predict-inferences.zarr/')

# merge with predict in order to get x,y info
merged = predict.drop_vars('pm25').merge(predict_inferences, join='inner')

# stack by sample, add (x,y), and unstack
merged = merged.stack(sample=['id','time'])\
               .reset_index('sample')\
               .set_coords(['x','y'])\
               .drop_vars('id')\
               .set_index(sample=['time','y','x'])\
               .unstack('sample')

# print
print(50*'-'+f'\n\n{merged}\n')

# plot
%matplotlib inline
plt.close()
merged.pm25.plot(col='time', col_wrap=6, vmin=0)
plt.show()